[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C61_Detection_Practice_Interview_Course/05_drills/05_interview_drills.ipynb)

# 05 · 面试演练题库（IoU / NMS / soft-NMS / mAP / 匈牙利 / Focal / GIoU 全部手写）

这是整批课程的收官 notebook。目标不是「跑通」，而是**让你能在白板上默写出来**。
纯 numpy + 标准库，CPU 秒级完成。

本 notebook 你会亲手实现（每个都配单元测试，含边界情况）：
1. **IoU** —— clip / 零面积保护 / 尺度不公平性的数值证据
2. **NMS** —— 稳定排序 / class-wise 的 offset 技巧
3. **soft-NMS** —— 线性与高斯衰减，以及「输出框变多」这个代价
4. **mAP** —— 完整流程 + **VOC 面积法 / VOC11 / COCO101 三种口径给出不同的数**
5. **匈牙利匹配** —— 自己写 $O(n^2m)$ 版本，与暴力枚举对拍，并给出**贪心失败的反例**
6. **Focal Loss** —— 数值稳定写法 + $\gamma$/$\alpha$ 分工的数值验证 + prior bias 初始化
7. **GIoU** —— 不相交时仍有梯度的证据，以及 $\mathrm{GIoU} \le \mathrm{IoU}$ 的性质检验
8. **答题结构检查器** —— 判断依据 → 方案 → 验证，机器检查你的回答
9. **自测题库** —— 22 道题，先看问题自己答，再 `reveal(id)` 对照

> 白板题考的从来不是「会不会写」，而是**你写代码时暴露的工程习惯**：
> 边界意识、复杂度敏感、以及知不知道这段代码在真实系统的哪个位置。

## 1 · IoU：所有检测题的地基

三个必答的边界情况：**不相交（必须 clip 到 0）/ 完全包含 / 退化框（零面积，分母保护）**。
写完再补一句尺度不公平性 —— 这是把 IoU 连到 TSR 小目标的关键。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def box_area(b):
    return np.clip(b[..., 2] - b[..., 0], 0, None) * np.clip(b[..., 3] - b[..., 1], 0, None)

def iou_1toN(box, boxes, eps=1e-9):
    """box:(4,) xyxy;  boxes:(N,4)  ->  (N,) IoU。假设：xyxy、不含 +1、像素坐标。"""
    box = np.asarray(box, dtype=float)
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)   # ★ 不 clip 会算出正的假 IoU
    union = box_area(box) + box_area(boxes) - inter
    return inter / np.maximum(union, eps)                            # ★ 分母保护，退化框不除零

A = np.array([0., 0., 10., 10.])
cases = [('完全重合', [0, 0, 10, 10], 1.0),
         ('完全不相交', [20, 20, 30, 30], 0.0),
         ('仅边界相接', [10, 10, 20, 20], 0.0),
         ('完全包含(小框在内)', [2, 2, 4, 4], 4.0 / 100),
         ('部分重叠', [1, 1, 11, 11], 81.0 / 119),
         ('退化框(零面积)', [5, 5, 5, 5], 0.0)]
for name, b, expect in cases:
    got = float(iou_1toN(A, [b])[0])
    print('  %-18s IoU = %.6f   (期望 %.6f)' % (name, got, expect))
    assert abs(got - expect) < 1e-12, (name, got, expect)

def shift_iou(size, shift):
    return float(iou_1toN([0, 0, size, size], [[shift, shift, size + shift, size + shift]])[0])

print('\n同样位移 2 px，IoU 随尺度的衰减（TSR 小目标的核心论据）：')
for s in [8, 16, 32, 64]:
    print('  %3d x %-3d 框位移 2px  ->  IoU %.4f' % (s, s, shift_iou(s, 2)))
assert abs(shift_iou(8, 2) - 36 / 92) < 1e-12      # 交 6x6=36, 并 64+64-36=92
assert abs(shift_iou(64, 2) - 3844 / 4348) < 1e-12
assert shift_iou(8, 2) < 0.40 and shift_iou(64, 2) > 0.88
print('\n✅ 8x8 掉到 0.39，64x64 还有 0.88 —— **同一个 IoU 阈值对不同尺度极不公平**。')

## 2 · NMS：稳定排序与 class-wise

两个必答细节：**`argsort` 默认不稳定**（同分时结果不可复现，会让回归测试随机失败）；
**class-wise 用 offset 技巧一次搞定**（torchvision `batched_nms` 的真实做法）。

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    """标准贪心 NMS。返回保留下标（按分数降序）。复杂度 O(N log N + N*K)。"""
    boxes = np.asarray(boxes, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind='stable')      # ★ 稳定排序：同分时可复现
    keep = []
    while order.size > 0:
        i = int(order[0]); keep.append(i)
        if order.size == 1:
            break
        rest = order[1:]
        ious = iou_1toN(boxes[i], boxes[rest])
        order = rest[ious <= iou_thr]               # ★ <= ：等于阈值时不抑制
    return np.array(keep, dtype=int)

def batched_nms(boxes, scores, labels, iou_thr=0.5):
    """class-wise NMS 的 offset 技巧：给不同类别的框加一个足够大的坐标偏移，
       使它们在数值上永不相交，然后跑一次 class-agnostic NMS。"""
    boxes = np.asarray(boxes, dtype=float)
    if boxes.size == 0:
        return np.array([], dtype=int)
    offset = np.asarray(labels, dtype=float) * (boxes.max() + 1.0)
    return nms(boxes + offset[:, None], scores, iou_thr)

B = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [50, 50, 60, 60]], dtype=float)
S = np.array([0.9, 0.8, 0.7])
print('IoU(0,1) = 81/119 = %.4f  > 0.5  ->  框 1 被抑制' % (81 / 119))
print('class-agnostic keep =', nms(B, S, 0.5).tolist())
assert nms(B, S, 0.5).tolist() == [0, 2]

# 同分可复现性
assert np.argsort(-np.array([0.9, 0.9, 0.7]), kind='stable').tolist() == [0, 1, 2]
assert nms(B, np.array([0.9, 0.9, 0.7]), 0.5).tolist() == [0, 2]

# class-wise：同样两个重叠框，属于不同类别时必须都保留
L_same, L_diff = np.array([0, 0, 0]), np.array([0, 1, 0])
assert batched_nms(B, S, L_same, 0.5).tolist() == [0, 2]
assert sorted(batched_nms(B, S, L_diff, 0.5).tolist()) == [0, 1, 2]
print('class-wise(同类) keep =', batched_nms(B, S, L_same, 0.5).tolist(),
      '  class-wise(异类) keep =', sorted(batched_nms(B, S, L_diff, 0.5).tolist()))
print('\n✅ 忘了 class-wise，一个「限速 60」会把重叠的「路面湿滑」删掉 —— 线上很难查的漏检。')

## 3 · soft-NMS：把「删除」换成「降分」

$f_{\text{linear}}(u) = 1-u \ (u \ge N_t)$，$f_{\text{gauss}}(u) = e^{-u^2/\sigma}$。
**代价必须一起说**：输出框变多（后面必须接 score 阈值）、多一个超参 $\sigma$、
固定工作点下的 precision 不一定更好。

In [ ]:
def soft_nms(boxes, scores, method='gaussian', sigma=0.5, iou_thr=0.5, score_thr=1e-3):
    """返回 (保留下标, 衰减后的分数)。method ∈ gaussian | linear | hard。"""
    boxes = np.asarray(boxes, dtype=float)
    s = np.asarray(scores, dtype=float).copy()
    remaining = list(range(len(boxes)))
    keep, kept_scores = [], []
    while remaining:
        i = max(remaining, key=lambda j: s[j])
        remaining.remove(i)
        keep.append(i); kept_scores.append(float(s[i]))
        if not remaining:
            break
        rest = np.array(remaining)
        ious = iou_1toN(boxes[i], boxes[rest])
        if method == 'linear':
            decay = np.where(ious >= iou_thr, 1.0 - ious, 1.0)
        elif method == 'gaussian':
            decay = np.exp(-(ious ** 2) / sigma)
        else:                                   # hard = 退化成标准 NMS
            decay = (ious <= iou_thr).astype(float)
        s[rest] = s[rest] * decay
        remaining = [j for j in remaining if s[j] > score_thr]
    return np.array(keep, dtype=int), np.array(kept_scores)

k_g, sc_g = soft_nms(B, S, 'gaussian', sigma=0.5)
k_h, _ = soft_nms(B, S, 'hard', iou_thr=0.5)
u01 = 81 / 119
print('gaussian  keep =', k_g.tolist(), '  scores =', np.round(sc_g, 4).tolist())
print('hard      keep =', k_h.tolist(), '  （与标准 NMS 一致）')
assert k_g.tolist() == [0, 2, 1], k_g
assert abs(sc_g[2] - 0.8 * np.exp(-(u01 ** 2) / 0.5)) < 1e-12
assert k_h.tolist() == nms(B, S, 0.5).tolist()
print('\n重复框的分数被压到 %.4f（原 0.80），但**它还在输出里**：' % sc_g[2])
print('  -> soft-NMS 后必须接 score 阈值，否则下游被一堆低分重复框淹没。')
print('✅ 适用判据：**你的重叠是「真实相邻目标」还是「同一目标的重复检测」**。')

## 4 · mAP：完整流程与三种口径

最常被忘的一条：**每个 GT 只能被匹配一次**（否则重复检测全算 TP，AP 虚高）。
第二常被忘：**recall 的分母是 GT 总数**，而且末尾要补 (recall=1, precision=0)，
否则漏检不受惩罚。

In [ ]:
def voc_ap(rec, prec, mode='area'):
    """mode ∈ area(VOC2010+) | 11point(VOC2007) | 101point(COCO)。"""
    mrec = np.concatenate([[0.0], np.asarray(rec, float), [1.0]])   # ★ 末尾补 1，漏检才受罚
    mpre = np.concatenate([[0.0], np.asarray(prec, float), [0.0]])
    for i in range(len(mpre) - 2, -1, -1):                          # ★ 单调包络：从右往左
        mpre[i] = max(mpre[i], mpre[i + 1])
    if mode == 'area':
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    n = {'11point': 11, '101point': 101}[mode]
    ts = np.linspace(0.0, 1.0, n)
    return float(np.mean([float(mpre[mrec >= t].max()) if np.any(mrec >= t) else 0.0
                          for t in ts]))

def match_and_curve(pred_boxes, pred_scores, gt_boxes, iou_thr=0.5):
    """单类：按分数降序判 TP/FP，返回 (rec, prec, tp, fp)。"""
    pred_boxes = np.asarray(pred_boxes, float)
    order = np.argsort(-np.asarray(pred_scores, float), kind='stable')
    gt = np.asarray(gt_boxes, float).reshape(-1, 4)
    used = np.zeros(len(gt), dtype=bool)
    tp = np.zeros(len(order)); fp = np.zeros(len(order))
    for r, i in enumerate(order):
        if len(gt) == 0:
            fp[r] = 1.0; continue
        ious = iou_1toN(pred_boxes[i], gt)
        j = int(np.argmax(ious))
        if ious[j] >= iou_thr and not used[j]:     # ★ 每个 GT 只匹配一次
            used[j] = True; tp[r] = 1.0
        else:
            fp[r] = 1.0
    ctp, cfp = np.cumsum(tp), np.cumsum(fp)
    rec = ctp / max(len(gt), 1)                    # ★ 分母是 GT 数
    prec = ctp / np.maximum(ctp + cfp, 1e-12)
    return rec, prec, tp, fp

GT = np.array([[0, 0, 10, 10], [50, 50, 60, 60], [100, 100, 120, 120]], float)
PB = np.array([[0, 0, 10, 10],            # TP：与 G0 IoU=1
               [1, 1, 11, 11],            # FP：G0 已被更高分的认领 -> 重复检测
               [51, 51, 61, 61],          # TP：与 G1 IoU=81/119=0.681
               [200, 200, 210, 210]], float)   # FP：纯背景
PS = np.array([0.9, 0.8, 0.7, 0.6])       # G2 从未被检出 -> recall 上限 2/3

rec, prec, tp, fp = match_and_curve(PB, PS, GT, 0.5)
print('TP/FP 序列 :', tp.astype(int).tolist(), fp.astype(int).tolist())
print('recall     :', np.round(rec, 4).tolist())
print('precision  :', np.round(prec, 4).tolist())
assert tp.tolist() == [1, 0, 1, 0]
assert np.allclose(rec, [1 / 3, 1 / 3, 2 / 3, 2 / 3])
assert np.allclose(prec, [1.0, 0.5, 2 / 3, 0.5])

ap_area, ap11, ap101 = (voc_ap(rec, prec, m) for m in ('area', '11point', '101point'))
print('\nAP（VOC 面积法）  = %.6f   期望 5/9  = %.6f' % (ap_area, 5 / 9))
print('AP（VOC2007 11点）= %.6f   期望 6/11 = %.6f  <- 系统性偏高' % (ap11, 6 / 11))
print('AP（COCO 101点）  = %.6f   期望 56/101 = %.6f' % (ap101, 56 / 101))
assert abs(ap_area - 5 / 9) < 1e-12
assert abs(ap11 - 6 / 11) < 1e-12
assert abs(ap101 - 56 / 101) < 1e-12
print('\n✅ **同一条 PR 曲线，三种口径给出三个不同的数** —— 所以报 mAP 必须报口径。')

## 5 · 匈牙利匹配：自己写一个（不许用 scipy）

DETR 的一对一分配。先记住**贪心失败的反例**（30 秒能写在白板上），
再写 $O(n^2 m)$ 的 Kuhn-Munkres（带对偶变量的增广路径版本）。

In [ ]:
from itertools import permutations

def hungarian(cost):
    """最小代价的一对一匹配。cost:(n,m) 且 n<=m（预测多于 GT 时请转置）。
       返回 (assign, total)，assign[i] = 第 i 行匹配到的列下标。O(n^2 m)。"""
    a = np.asarray(cost, dtype=float)
    n, m = a.shape
    assert n <= m, '行数必须 <= 列数'
    INF = float('inf')
    u = [0.0] * (n + 1); v = [0.0] * (m + 1)
    p = [0] * (m + 1); way = [0] * (m + 1)
    for i in range(1, n + 1):
        p[0] = i; j0 = 0
        minv = [INF] * (m + 1); used = [False] * (m + 1)
        while True:                                  # 找一条增广路
            used[j0] = True
            i0 = p[j0]; delta = INF; j1 = 0
            for j in range(1, m + 1):
                if not used[j]:
                    cur = a[i0 - 1, j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur; way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(m + 1):                   # 更新对偶变量（势）
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:                                    # 沿增广路翻转匹配
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    assign = np.full(n, -1, dtype=int)
    for j in range(1, m + 1):
        if p[j] > 0:
            assign[p[j] - 1] = j - 1
    return assign, float(sum(a[i, assign[i]] for i in range(n)))

def greedy_match(cost):
    """贪心：每次取全局最小，然后划掉该行该列。用来演示它为什么不行。"""
    a = np.array(cost, dtype=float); n, m = a.shape
    assign = np.full(n, -1, dtype=int); total = 0.0
    for _ in range(n):
        i, j = np.unravel_index(int(np.argmin(a)), a.shape)
        assign[i] = j; total += float(a[i, j])
        a[i, :] = np.inf; a[:, j] = np.inf
    return assign, total

# —— 贪心失败的反例（背下来，30 秒可写）——
CE = np.array([[1., 2.], [2., 100.]])
ag, tg = greedy_match(CE); ah, th = hungarian(CE)
print('反例代价矩阵 [[1,2],[2,100]]')
print('  贪心  : assign =', ag.tolist(), ' 总代价 =', tg, '  <- 先吃掉 1，把行 1 逼到 100')
print('  匈牙利: assign =', ah.tolist(), ' 总代价 =', th)
assert tg == 101.0 and th == 4.0 and ah.tolist() == [1, 0]

C3 = np.array([[4., 1., 3.], [2., 0., 5.], [3., 2., 2.]])
a3, t3 = hungarian(C3)
assert a3.tolist() == [1, 0, 2] and abs(t3 - 5.0) < 1e-12, (a3, t3)
print('\n经典 3x3 : assign =', a3.tolist(), ' 总代价 =', t3)

# —— 与暴力枚举对拍 ——
for _ in range(200):                                   # 方阵
    C = rng.integers(0, 20, size=(5, 5)).astype(float)
    _, tot = hungarian(C)
    best = min(sum(C[i, pm[i]] for i in range(5)) for pm in permutations(range(5)))
    assert abs(tot - best) < 1e-9
for _ in range(100):                                   # 长方阵（预测数 > GT 数的典型情形）
    C = rng.integers(0, 20, size=(3, 6)).astype(float)
    _, tot = hungarian(C)
    best = min(sum(C[i, pm[i]] for i in range(3)) for pm in permutations(range(6), 3))
    assert abs(tot - best) < 1e-9
print('✅ 300 组随机矩阵与暴力枚举完全一致（含 3x6 长方阵：DETR 里 N 个预测 > M 个 GT）。')

## 6 · Focal Loss：$\gamma$ 与 $\alpha$ 各管什么

$\mathrm{FL}(p_t) = -\alpha_t (1-p_t)^{\gamma}\log p_t$。
两个白板陷阱：**数值稳定**（直接写 `-log(sigmoid(x))` 会溢出）与
**prior bias 初始化** $b_0 = -\log((1-\pi)/\pi)$。

In [ ]:
def sigmoid(x):
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(x, dtype=float)))   # tanh 形式天然不溢出

def bce_with_logits(z, y):
    """数值稳定的 BCE： max(z,0) - z*y + log(1+exp(-|z|))"""
    z = np.asarray(z, dtype=float); y = np.asarray(y, dtype=float)
    return np.maximum(z, 0.0) - z * y + np.log1p(np.exp(-np.abs(z)))

def focal_loss_with_logits(z, y, alpha=0.25, gamma=2.0):
    z = np.asarray(z, dtype=float); y = np.asarray(y, dtype=float)
    p = sigmoid(z)
    p_t = p * y + (1.0 - p) * (1.0 - y)
    a_t = alpha * y + (1.0 - alpha) * (1.0 - y)
    return a_t * (1.0 - p_t) ** gamma * bce_with_logits(z, y)

# ① gamma=0 时退化成加权 BCE
zz = np.linspace(-5, 5, 21); yy = (zz > 0).astype(float)
assert np.allclose(focal_loss_with_logits(zz, yy, alpha=0.5, gamma=0.0), 0.5 * bce_with_logits(zz, yy))
# ② 稳定实现 == 朴素实现（在不溢出的范围内）
p_naive = 1.0 / (1.0 + np.exp(-zz))
naive = -(yy * np.log(p_naive) + (1 - yy) * np.log(1 - p_naive))
assert np.allclose(bce_with_logits(zz, yy), naive)
# ③ 极端 logit 不产生 inf/nan
ex = np.array([-100., -50., 0., 50., 100.])
for tgt in (0.0, 1.0):
    assert np.all(np.isfinite(focal_loss_with_logits(ex, tgt))), tgt

# ④ gamma 的作用：压易不压难
def ratio(z, y, g=2.0):
    return float(focal_loss_with_logits(z, y, 0.25, g) / focal_loss_with_logits(z, y, 0.25, 0.0))
r_easy, r_hard = ratio(-6.0, 0.0), ratio(0.0, 0.0)
print('易分负样本 (logit=-6): 损失被压到 gamma=0 时的 %.3e 倍' % r_easy)
print('难样本     (logit= 0): 损失被压到 gamma=0 时的 %.4f 倍' % r_hard)
print('  -> 两者相差 %.0f 倍，这个比例差就是 Focal Loss 的全部作用' % (r_hard / r_easy))
assert r_easy < 1e-5 and abs(r_hard - 0.25) < 1e-12

# ⑤ 梯度贡献（数值微分）
def num_grad(f, z, h=1e-5):
    return (f(z + h) - f(z - h)) / (2 * h)
g0 = num_grad(lambda t: float(focal_loss_with_logits(t, 0.0, 0.25, 0.0)), -6.0)
g2 = num_grad(lambda t: float(focal_loss_with_logits(t, 0.0, 0.25, 2.0)), -6.0)
print('\n易分负样本的梯度：gamma=0 时 %.3e，gamma=2 时 %.3e，压到 %.2e 倍' % (g0, g2, g2 / g0))
assert abs(g2 / g0) < 1e-4

# ⑥ prior bias 初始化
pi = 0.01
b0 = -np.log((1 - pi) / pi)
print('prior bias b0 = -log((1-pi)/pi) = %.4f  ->  sigmoid(b0) = %.6f' % (b0, float(sigmoid(b0))))
assert abs(float(sigmoid(b0)) - pi) < 1e-12
print('✅ 不做这个初始化，训练最初几个 iteration 会被海量负样本的损失主导而发散。')

## 7 · GIoU：不相交时仍然有梯度

$\mathrm{GIoU} = \mathrm{IoU} - \dfrac{|C \setminus (A\cup B)|}{|C|}$，值域 $[-1,1]$，
且恒有 $\mathrm{GIoU} \le \mathrm{IoU}$。

In [ ]:
def giou_1toN(box, boxes, eps=1e-9):
    box = np.asarray(box, dtype=float)
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    union = box_area(box) + box_area(boxes) - inter
    iou = inter / np.maximum(union, eps)
    cx1 = np.minimum(box[0], boxes[:, 0]); cy1 = np.minimum(box[1], boxes[:, 1])
    cx2 = np.maximum(box[2], boxes[:, 2]); cy2 = np.maximum(box[3], boxes[:, 3])
    area_c = np.clip(cx2 - cx1, 0, None) * np.clip(cy2 - cy1, 0, None)   # 最小包围框
    return iou - (area_c - union) / np.maximum(area_c, eps)

A2 = np.array([0., 0., 10., 10.])
print('  两框重合          IoU=%.3f  GIoU=%.4f' % (iou_1toN(A2, [[0, 0, 10, 10]])[0],
                                                  giou_1toN(A2, [[0, 0, 10, 10]])[0]))
for d in [20, 40, 90]:
    b = [[d, d, d + 10, d + 10]]
    print('  相隔 %2d 像素      IoU=%.3f  GIoU=%.4f  <- IoU 全是 0，GIoU 还在变'
          % (d, iou_1toN(A2, b)[0], giou_1toN(A2, b)[0]))
assert abs(float(giou_1toN(A2, [[0, 0, 10, 10]])[0]) - 1.0) < 1e-12
assert abs(float(giou_1toN(A2, [[20, 20, 30, 30]])[0]) - (-700 / 900)) < 1e-12
assert abs(float(giou_1toN(A2, [[90, 90, 100, 100]])[0]) - (-9800 / 10000)) < 1e-12
gs = [float(giou_1toN(A2, [[d, d, d + 10, d + 10]])[0]) for d in [20, 40, 90]]
assert gs[0] > gs[1] > gs[2], gs                       # 距离越远越差（单调）

# 性质检验：-1 <= GIoU <= IoU <= 1
for _ in range(500):
    c1, s1 = rng.uniform(0, 100, 2), rng.uniform(1, 40, 2)
    c2, s2 = rng.uniform(0, 100, 2), rng.uniform(1, 40, 2)
    a = np.array([c1[0], c1[1], c1[0] + s1[0], c1[1] + s1[1]])
    b = np.array([[c2[0], c2[1], c2[0] + s2[0], c2[1] + s2[1]]])
    g, u = float(giou_1toN(a, b)[0]), float(iou_1toN(a, b)[0])
    assert -1.0 - 1e-9 <= g <= u + 1e-9 and u <= 1.0 + 1e-9
print('\n✅ 500 组随机框满足 -1 <= GIoU <= IoU <= 1；不相交时 IoU 恒 0（无梯度），GIoU 仍单调。')

## 8 · 答题结构检查器：判断依据 → 方案 → 验证

用关键词指纹检查一段口头回答有没有覆盖三段，以及**顺序对不对**。
把你自己的答案粘进来跑一遍 —— 多数人的第 ① 段和第 ③ 段是空的。

In [ ]:
STRUCTURE_MARKERS = [
    ('① 判断依据', ['我先看', '先确认', '依据', '瓶颈', '归因', '分解', '排除', '假设']),
    ('② 方案',     ['我会', '做法', '方案', '分三层', '第一步', '优先', '先做']),
    ('③ 验证',     ['验证', '切片', '种子', '门禁', '消融', '灰度', '显著', 'A/B']),
]

def check_answer_structure(text):
    """返回 (命中的段, 缺失的段, 顺序是否正确)。"""
    pos = {}
    for name, kws in STRUCTURE_MARKERS:
        idxs = [text.find(k) for k in kws if k in text]
        if idxs:
            pos[name] = min(idxs)
    names = [n for n, _ in STRUCTURE_MARKERS]
    hit = [n for n in names if n in pos]
    missing = [n for n in names if n not in pos]
    ordered = all(pos[hit[i]] < pos[hit[i + 1]] for i in range(len(hit) - 1))
    return hit, missing, ordered

GOOD = ('我先看误差分解：Miss 占 62% 且集中在小目标桶，所以我排除了换 backbone 这条路。'
        '方案分三层：数据侧先做 copy-paste 和提分辨率，损失侧再考虑 scale-aware 加权，'
        '架构侧的 P2 层留到最后，因为延迟代价大。'
        '验证上我会跑 3 个种子做配对检验，按尺寸分桶看切片，并让旧场景过回归门禁。')
BAD = '我会用 copy-paste 和重采样，再加一个 Focal Loss，然后换个更大的 backbone。'

for label, ans in [('好答案', GOOD), ('差答案', BAD)]:
    hit, missing, ordered = check_answer_structure(ans)
    print('%s：命中 %s' % (label, hit))
    print('        缺失 %s   顺序正确 = %s' % (missing if missing else '无', ordered))
assert check_answer_structure(GOOD) == (['① 判断依据', '② 方案', '③ 验证'], [], True)
h, m_, o = check_answer_structure(BAD)
assert h == ['② 方案'] and m_ == ['① 判断依据', '③ 验证'] and o is True
print('\n✅ 差答案不是「错」，是**只有中间一段**：听不出你为什么这么选，也听不出你怎么证明它有用。')

## 9 · 自测题库：22 道题

用法：`ask(id)` 只看题目，自己出声答一遍（**计时 60 秒**），再 `reveal(id)` 对照要点。
要点不是标准答案，是**你的回答里必须出现的关键词**——少一个就是一个失分点。

In [ ]:
from collections import Counter

QUIZ = [
 {'id': 1,  'cat': '白板', 'q': '手写 IoU，并说出三个必须处理的边界情况。',
  'key': ['clip 到 0（不相交时不能算出正 IoU）', '分母 max(union, eps) 防退化框除零',
          '+1 的历史包袱（VOC 用 x2-x1+1，COCO 不加）', '8x8 框位移 2px -> IoU 0.39 的尺度不公平']},
 {'id': 2,  'cat': '白板', 'q': '手写 NMS；为什么必须用稳定排序？',
  'key': ['argsort 默认不稳定，同分时结果不可复现 -> 回归测试 flaky',
          'class-wise vs class-agnostic', 'offset 技巧（torchvision batched_nms）',
          'O(N log N + N*K)，部署要先 top-k 预筛把 N 钉死']},
 {'id': 3,  'cat': '白板', 'q': 'soft-NMS 是什么？它的代价是什么？',
  'key': ['把硬删除换成降分：线性 1-u / 高斯 exp(-u^2/sigma)',
          '代价：输出框变多（必须接 score 阈值）、多一个超参、固定工作点 precision 不一定更好',
          '适用判据：重叠是「真实相邻目标」还是「重复检测」']},
 {'id': 4,  'cat': '白板', 'q': '完整推导 mAP 的计算流程。',
  'key': ['按 score 降序 -> IoU 匹配且**每个 GT 只匹配一次** -> 累积 P/R',
          'recall 分母是 GT 总数；末尾补 (1, 0) 让漏检受罚',
          '单调包络（从右往左取后缀最大）-> 求面积',
          'VOC11 / VOC 面积法 / COCO 101 点 + IoU 0.5:0.05:0.95 十档']},
 {'id': 5,  'cat': '白板', 'q': '匈牙利匹配怎么做？为什么贪心不行？代价含哪几项？',
  'key': ['一对一最优匹配，O(n^3)；反例 [[1,2],[2,100]]：贪心 101 vs 最优 4',
          '代价 = -分类**概率**（不是 log）+ L1(归一化框) + (-GIoU)',
          '一对一 -> 训练期压制重复 -> 推理不需要 NMS',
          '匹配不稳定性 -> DN-DETR / DINO 的动机']},
 {'id': 6,  'cat': '白板', 'q': '手写 Focal Loss；gamma 和 alpha 各控制什么？',
  'key': ['FL = -alpha_t (1-p_t)^gamma log p_t',
          'gamma 按**难度**加权（压易不压难），alpha 按**类别**加权',
          '只有 alpha 不够：易分负样本数量级 1e4，累加仍压倒前景',
          '数值稳定：max(z,0) - z*y + log1p(exp(-|z|))；prior bias b0 = -log((1-pi)/pi)']},
 {'id': 7,  'cat': '白板', 'q': 'GIoU 解决什么问题？DIoU / CIoU / NWD 各加了什么？',
  'key': ['IoU 在不相交时梯度为 0，模型不知道差一点和差很远的区别',
          'GIoU 减最小包围框的空白占比；DIoU 加中心距离（也可用于 NMS）；CIoU 加长宽比',
          'NWD：框建模成 2D 高斯 + Wasserstein 距离，**对尺度不敏感**，小目标的解',
          '选哪个应由误差分解驱动：损失在 Miss 上时换 IoU 变体没用']},
 {'id': 8,  'cat': '架构', 'q': 'DETR 为什么不需要 NMS？',
  'key': ['一对一匈牙利匹配把去重提前到训练期', '与 Transformer 无关：YOLOv10 的一致双分配同样做到',
          '代价：监督稀疏 + 匹配翻转 -> 500 epoch 才收敛']},
 {'id': 9,  'cat': '架构', 'q': 'RT-DETR 凭什么比 YOLO 快？',
  'key': ['无 NMS -> 端到端延迟变常数（**均值 vs 方差要分开谈**）',
          'AIFI 只在 S5：8400 token 全局 attention 贵 205 倍；交叉点 HW = 2.5d',
          'CCFF 用卷积做跨尺度融合', 'query selection 决定 decoder 起跑线',
          'decoder 层数可调 -> 一份权重多档速度']},
 {'id': 10, 'cat': '架构', 'q': '标签分配的演进讲一下。',
  'key': ['静态 IoU 阈值 -> ATSS（均值+标准差自适应）-> OTA/SimOTA（最优传输 + dynamic-k）',
          '-> TaskAligned（t = s^a * u^b）-> RTMDet soft label -> 一对一',
          '主线：从几何规则固定分配 走向 按模型当前状态动态分配',
          '对小目标与密集场景影响最大']},
 {'id': 11, 'cat': '架构', 'q': 'anchor-free 的本质是什么？',
  'key': ['换了正样本的定义方式，不是删掉一个组件', '好处：去超参、无 IoU 阈值的尺度偏见、输出通道少',
          'ATSS 证明：性能差距主要来自分配方式而不是有没有 anchor']},
 {'id': 12, 'cat': '架构', 'q': 'FPN 的层级分配规则？小目标为什么会失效？',
  'key': ['k = k0 + log2(sqrt(wh)/224)，k0=4',
          '检测头通常只到 P3(stride 8)，小目标全挤在 P3；8px 目标只占 1 个格子',
          '加 P2 代价：token 数 x4，显存/算力按分辨率平方涨',
          '便宜的替代：提分辨率 / 两级级联 / 改分配与度量']},
 {'id': 13, 'cat': '数据与评测', 'q': '交通标志类别极度长尾，你怎么处理？',
  'key': ['**先澄清：实例级还是图像级长尾**', '数据侧：copy-paste / 定向挖掘 / repeat factor r=max(1,sqrt(t/f))',
          '损失侧：Focal / CB Loss / EQL', '推理侧：logit adjustment、解耦训练（零训练成本）',
          '架构侧：两级架构（检测粗类 + crop 分类）', '评测侧：按频段分桶（最先做）']},
 {'id': 14, 'cat': '数据与评测', 'q': 'mAP 不够用怎么办？怎么设计 TSR 评测？',
  'key': ['mAP 五不足：按类平均掩盖关键类 / 与距离无关 / 无时序 / 不反映 FP 绝对量 / 与下游脱节',
          '加：分桶指标、代价敏感指标、FP per km、时序稳定性（首检距离/闪烁率）',
          '每个失效模式一个回归小集 + 门禁规则']},
 {'id': 15, 'cat': '数据与评测', 'q': '+0.3 mAP 算提升吗？',
  'key': ['先问三件事：几个种子 / 同口径同工作点吗 / 分场景切片什么样',
          '检测任务单种子波动典型 ±0.2-0.5，+0.3 落在噪声里',
          '功效分析：sigma=0.35 时配对设计约需 8-10 个种子才有 80% 把握']},
 {'id': 16, 'cat': '数据与评测', 'q': '线上某类标志突然漏检，排查流程？',
  'key': ['第一刀：离线能不能复现 -> 复现不了就是部署一致性问题',
          '第二刀：降 score 阈值看框在不在 -> 在但分低是信心问题，没有是分配/尺度问题',
          '然后分桶归因 + 混淆矩阵', '原则：每一步砍掉一半假设空间']},
 {'id': 17, 'cat': '部署', 'q': 'INT8 后 mAP 掉 3 个点，怎么排查？',
  'key': ['先排除非量化因素：FP32 engine 对得上 PyTorch 吗？预处理一致吗？',
          '校准集诊断（**最高频的真实原因**）：覆盖夜间/逆光/雨天了吗？',
          '换校准算法：MinMax -> Entropy/KL -> Percentile',
          '逐层敏感度分析 -> 混合精度 / per-channel -> 最后才 QAT',
          '必须按尺寸分桶看：小目标掉得更多']},
 {'id': 18, 'cat': '部署', 'q': '离线 0.82 车上像 0.6，怎么查？',
  'key': ['三层二分：预处理 / 模型本体 / 后处理，逐层 dump 对拍',
          '预处理：BGR/RGB、resize 语义（align_corners、INTER_AREA）、letterbox 分歧',
          '模型：opset、动态 shape profile、FP16 溢出、**静默回退（分区数 > 1）**',
          '后处理：letterbox 逆变换（框整体偏移的头号原因）、NMS 差异、sigmoid 位置']},
 {'id': 19, 'cat': '部署', 'q': '怎么正确测延迟？为什么关心 p99？',
  'key': ['warmup / 显式同步 / 取分布不取均值 / 报 p50 与 p99',
          '要用目标数差异大的真实帧，不能同一张图跑一千次',
          '车端特有：功耗降频、多任务抢占、H2D/D2H 拷贝',
          '报延迟必须写口径：batch、卡型、含不含预处理与 NMS、精度']},
 {'id': 20, 'cat': '系统设计', 'q': '设计一个 TSR 的数据-模型-评测-部署闭环。',
  'key': ['触发（不确定性/一致性/规则/事件）-> 挖掘（嵌入检索 + 场景标签 + 去重多样性）',
          '-> 标注（层次标签 + 一致性抽检）-> 训练（跟踪 + 配比）',
          '-> 分场景评测 -> 回归门禁（含多重比较校正）-> 影子/灰度 -> 回流',
          '闭环的度量 = 从发现问题到修复上线的周期时间',
          '触发器本身有偏：会漏掉「自信地错」的那类']},
 {'id': 21, 'cat': '系统设计', 'q': '把夜间场景的 recall 提 10 个点，你会怎么做？',
  'key': ['先做三件不花钱的：夜间桶 TIDE 分解 / 混淆矩阵 / 训练集夜间占比',
          '再分层：数据侧（挖掘 + 低光合成 + copy-paste）/ 采样损失侧 / 模型侧 / 评测侧',
          '主动说「我不会一上来换更大模型」，夜间通常不是容量问题']},
 {'id': 22, 'cat': '系统设计', 'q': 'TSR 的输出怎么接入 VLA / 下游规控？',
  'key': ['三种融合层次：符号/文本级、特征级、中间表示级（BEV query）',
          'schema 必须含**置信度**：不传等于强迫下游把所有检测当真',
          '标志语义 -> 可执行约束；**作用域与生命周期**最容易出错',
          '冲突优先级；VLA 之外必须有规则层与安全兜底']},
]

BY_ID = {q['id']: q for q in QUIZ}

def ask(qid):
    q = BY_ID[qid]
    print('[%s] Q%d. %s' % (q['cat'], q['id'], q['q']))
    print('    （计时 60 秒，出声答一遍，再 reveal(%d)）' % qid)

def reveal(qid):
    q = BY_ID[qid]
    print('[%s] Q%d 要点（少一条就是一个失分点）:' % (q['cat'], q['id']))
    for k in q['key']:
        print('   ·', k)

def drill(n=3, seed=None):
    r = np.random.default_rng(seed)
    for i in r.choice([q['id'] for q in QUIZ], size=n, replace=False):
        ask(int(i)); print()

CATS = Counter(q['cat'] for q in QUIZ)
print('题库分布：', dict(CATS), ' 总计', len(QUIZ), '题\n')
assert len(QUIZ) == 22
assert dict(CATS) == {'白板': 7, '架构': 5, '数据与评测': 4, '部署': 3, '系统设计': 3}
assert len(BY_ID) == 22 and all(len(q['key']) >= 3 for q in QUIZ)
drill(2, seed=7)
reveal(5)
print('\n✅ 用法：drill(3) 随机抽题 -> 自己答 -> reveal(id) 对照。**每天 10 分钟，一周过三轮。**')

## ✏️ 练习 1：向量化的 IoU 矩阵

实现 `iou_matrix(A, B)`：`A` 是 `(N,4)`、`B` 是 `(M,4)`，返回 `(N,M)` 的 IoU 矩阵。

> 面试官问完标量 IoU，**下一句几乎一定是「写个 N×M 的版本」** ——
> 因为 NMS、标签分配、匈牙利代价矩阵全都要它。
> 提示：`A[:, None, :]` 与 `B[None, :, :]` 广播。

In [ ]:
def iou_matrix(A, B, eps=1e-9):
    # TODO: 用广播，不许写双重 for 循环
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Aa = np.array([[0, 0, 10, 10], [50, 50, 60, 60]], float)
Bb = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30]], float)
Mx = iou_matrix(Aa, Bb)
assert Mx.shape == (2, 3), Mx.shape
assert abs(Mx[0, 0] - 1.0) < 1e-12
assert abs(Mx[0, 1] - 81 / 119) < 1e-12
assert abs(Mx[0, 2] - 0.0) < 1e-12
assert np.allclose(Mx[1], [0.0, 0.0, 0.0])
# 与逐行 iou_1toN 在随机数据上一致
for _ in range(50):
    c1, s1 = rng.uniform(0, 100, (6, 2)), rng.uniform(1, 40, (6, 2))
    c2, s2 = rng.uniform(0, 100, (9, 2)), rng.uniform(1, 40, (9, 2))
    P = np.concatenate([c1, c1 + s1], 1); Q = np.concatenate([c2, c2 + s2], 1)
    ref = np.stack([iou_1toN(P[i], Q) for i in range(len(P))])
    assert np.allclose(iou_matrix(P, Q), ref)
print(np.round(Mx, 4))
print('✅ 练习 1 通过：50 组随机数据与逐行实现完全一致（6x9 矩阵）。')

## ✏️ 练习 2：class-wise NMS（offset 技巧）

实现 `nms_class_wise(boxes, scores, labels, iou_thr)`：不同类别的框**互不抑制**。
可以调用已有的 `nms()`，但**不许调用 `batched_nms`**，要自己写 offset。

> 关键：offset 必须大于坐标的最大跨度，否则不同类别的框仍会重叠。

In [ ]:
def nms_class_wise(boxes, scores, labels, iou_thr=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert sorted(nms_class_wise(B, S, np.array([0, 0, 0]), 0.5).tolist()) == [0, 2]
assert sorted(nms_class_wise(B, S, np.array([0, 1, 0]), 0.5).tolist()) == [0, 1, 2]
# 与 batched_nms 在随机数据上等价
for _ in range(50):
    n = 30
    c, wh = rng.uniform(0, 200, (n, 2)), rng.uniform(5, 60, (n, 2))
    bx = np.concatenate([c, c + wh], 1)
    sc = rng.uniform(0, 1, n)
    lb = rng.integers(0, 4, n)
    assert sorted(nms_class_wise(bx, sc, lb, 0.5).tolist()) == \
           sorted(batched_nms(bx, sc, lb, 0.5).tolist())
print('同类   ->', sorted(nms_class_wise(B, S, np.array([0, 0, 0]), 0.5).tolist()))
print('异类   ->', sorted(nms_class_wise(B, S, np.array([0, 1, 0]), 0.5).tolist()))
print('✅ 练习 2 通过：50 组随机数据（4 个类别、30 个框）与 batched_nms 完全一致。')

## ✏️ 练习 3：COCO mAP@[.5:.95]

实现 `coco_map(pred_boxes, pred_scores, gt_boxes, thrs=None)`：
对 10 个 IoU 阈值（`np.linspace(0.5, 0.95, 10)`）分别用 **`voc_ap(..., 'area')`** 求 AP，
返回 `(平均 AP, 每个阈值的 AP 列表)`。

> 这题的教学点：**AP 随 IoU 阈值单调不增**，
> 所以 COCO 口径强烈奖励定位精度 —— 而小目标的 IoU 天生低，
> 用 COCO 口径会把小目标的分数压得很低（C57 模块 01）。

In [ ]:
def coco_map(pred_boxes, pred_scores, gt_boxes, thrs=None):
    # TODO: 复用 match_and_curve + voc_ap
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# （沿用第 4 节的 PB / PS / GT）
mAP, aps = coco_map(PB, PS, GT)
for t, a in zip(np.linspace(0.5, 0.95, 10), aps):
    print('  IoU=%.2f  AP=%.6f' % (t, a))
assert len(aps) == 10
assert all(aps[i] >= aps[i + 1] - 1e-12 for i in range(9)), 'AP 必须随阈值单调不增'
assert abs(aps[0] - 5 / 9) < 1e-12          # t<=0.68：P2 与 G1 的 IoU=81/119 仍算 TP
assert abs(aps[-1] - 1 / 3) < 1e-12         # t>=0.70：P2 变成 FP
assert abs(mAP - 19 / 45) < 1e-12
print('\nmAP@[.5:.95] = %.6f  （= 19/45）；而 mAP@0.5 = %.6f' % (mAP, aps[0]))
print('✅ 练习 3 通过：**同一组预测，COCO 口径比 VOC 口径低了 %.1f%%** —— 报数必须报口径。'
      % (100 * (1 - mAP / aps[0])))

## ✏️ 练习 4：DETR 的匹配代价矩阵

实现 `detr_cost_matrix(probs, pred_boxes, gt_labels, gt_boxes, w_cls, w_l1, w_giou)`：

$$\mathcal{C}_{i,j} = -w_{cls}\,\hat p_i(c_j) \;+\; w_{L1}\,\lVert \hat b_i - b_j\rVert_1 \;+\; w_{giou}\,(-\mathrm{GIoU}(\hat b_i, b_j))$$

三个必须记住的细节：**①分类项用概率不用 log**（用 log 会让分类项压倒框项）；
**②框必须归一化到 [0,1]**；**③预测数 N > GT 数 M 时，求匹配要转置**（`hungarian` 要求行数 ≤ 列数）。

In [ ]:
def detr_cost_matrix(probs, pred_boxes, gt_labels, gt_boxes,
                     w_cls=1.0, w_l1=5.0, w_giou=2.0):
    """probs:(N,C)  pred_boxes:(N,4) 归一化 xyxy
       gt_labels:(M,)  gt_boxes:(M,4)  ->  cost:(N,M)"""
    # TODO: 复用 giou_1toN
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
PROBS = np.array([[0.10, 0.80], [0.70, 0.20], [0.05, 0.05]])
PBOX = np.array([[0.0, 0.0, 0.1, 0.1], [0.5, 0.5, 0.6, 0.6], [0.8, 0.8, 0.9, 0.9]])
GLAB = np.array([1, 0])
GBOX = np.array([[0.0, 0.0, 0.1, 0.1], [0.5, 0.5, 0.6, 0.6]])

C = detr_cost_matrix(PROBS, PBOX, GLAB, GBOX)
assert C.shape == (3, 2), C.shape
assert abs(C[0, 0] - (-0.8 + 0.0 - 2.0)) < 1e-9, C[0, 0]       # 完美匹配：-p -0 -2*1
assert abs(C[1, 1] - (-0.7 + 0.0 - 2.0)) < 1e-9, C[1, 1]
assert abs(C[0, 1] - (-0.1 + 5 * 2.0 + 2 * 0.34 / 0.36)) < 1e-9, C[0, 1]
assert abs(C[2, 1] - (-0.05 + 5 * 1.2 + 2 * 0.14 / 0.16)) < 1e-9, C[2, 1]
print('代价矩阵（行=预测，列=GT）：')
print(np.round(C, 4))

assign, total = hungarian(C.T)      # ★ 转置：GT 作为行，因为 M(2) <= N(3)
print('\nGT -> 预测 的匹配：', assign.tolist(), '  总代价 =', round(total, 4))
assert assign.tolist() == [0, 1], assign
assert abs(total - (-5.5)) < 1e-9
print('✅ 练习 4 通过：第 3 个预测没有匹配到任何 GT -> 它会被推向 no-object 类。')
print('   这就是「一对一匹配在训练期压制重复」的全部机制。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def iou_matrix(A, B, eps=1e-9):
    A = np.asarray(A, float).reshape(-1, 4)[:, None, :]      # (N,1,4)
    B = np.asarray(B, float).reshape(-1, 4)[None, :, :]      # (1,M,4)
    x1 = np.maximum(A[..., 0], B[..., 0]); y1 = np.maximum(A[..., 1], B[..., 1])
    x2 = np.minimum(A[..., 2], B[..., 2]); y2 = np.minimum(A[..., 3], B[..., 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    union = box_area(A) + box_area(B) - inter
    return inter / np.maximum(union, eps)

In [ ]:
# 练习 2 参考答案
def nms_class_wise(boxes, scores, labels, iou_thr=0.5):
    boxes = np.asarray(boxes, dtype=float)
    if boxes.size == 0:
        return np.array([], dtype=int)
    span = float(boxes.max() - boxes.min()) + 1.0       # offset 必须 > 坐标跨度
    shifted = boxes + (np.asarray(labels, dtype=float) * span)[:, None]
    return nms(shifted, scores, iou_thr)

In [ ]:
# 练习 3 参考答案
def coco_map(pred_boxes, pred_scores, gt_boxes, thrs=None):
    if thrs is None:
        thrs = np.linspace(0.5, 0.95, 10)
    aps = []
    for t in thrs:
        rec_t, prec_t, _, _ = match_and_curve(pred_boxes, pred_scores, gt_boxes, float(t))
        aps.append(voc_ap(rec_t, prec_t, 'area'))
    return float(np.mean(aps)), aps

In [ ]:
# 练习 4 参考答案
def detr_cost_matrix(probs, pred_boxes, gt_labels, gt_boxes,
                     w_cls=1.0, w_l1=5.0, w_giou=2.0):
    probs = np.asarray(probs, float)
    pred_boxes = np.asarray(pred_boxes, float).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, float).reshape(-1, 4)
    gt_labels = np.asarray(gt_labels, int)
    cls_cost = -probs[:, gt_labels]                                   # (N,M) 用概率不用 log
    l1_cost = np.abs(pred_boxes[:, None, :] - gt_boxes[None, :, :]).sum(-1)
    giou = np.stack([giou_1toN(pred_boxes[i], gt_boxes) for i in range(len(pred_boxes))])
    return w_cls * cls_cost + w_l1 * l1_cost + w_giou * (-giou)

---
## 🧪 真实工程胶囊：面试前 72 小时执行清单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# TSR / 2D 检测岗 面试前 72 小时执行清单
# ══════════════════════════════════════════════════════════════════════

# ── T-72h ── 白板肌肉记忆（**不看本 notebook，白纸手写**）
#   IoU / NMS / mAP 各默写一遍，每遍写完自己列边界情况：
#     IoU : clip 到 0 / 分母保护 / +1 的历史包袱 / N×M 广播版
#     NMS : 稳定排序 / class-wise offset / <= 还是 < / top-k 预筛
#     mAP : 每个 GT 只匹配一次 / recall 分母 / 末尾补(1,0) / 单调包络方向
#   通过条件：三题都能在 8 分钟内写完并自己跑通两个例子
#   加练：匈牙利（能写出贪心反例 [[1,2],[2,100]] 即可）、Focal（稳定写法 + prior bias）

# ── T-48h ── 关键数字卡片（一页 A4，面试前 1 小时只看这张）
#   · 8x8 框位移 2px -> IoU 0.39；64x64 -> 0.88      （尺度不公平）
#   · AIFI 只在 S5：8400 token 全局 attention 贵 205x；交叉点 HW = 2.5d
#   · NMS 是 O(NK)，log-log 斜率约 1.7               （延迟方差）
#   · repeat factor r_c = max(1, sqrt(t/f_c))         （长尾重采样）
#   · 级联召回 = 检测召回 x 分类准确率                （两级架构）
#   · px = f*S/Z：60m 外 60cm 牌子 @1920x1080/60度 ≈ 17px
#   · 检测任务单种子 mAP 波动 ±0.2-0.5                （+0.3 是噪声）
#   · FL = -alpha_t (1-p_t)^gamma log p_t；gamma 管难度、alpha 管类别
#   · prior bias b0 = -log((1-pi)/pi)，pi=0.01

# ── T-24h ── 题库三轮
#   drill(5) x 3 轮，每题计时 60 秒出声答，reveal 对照要点
#   把答不全的题标出来，只复习这些
#   通过条件：22 题里至少 18 题能覆盖 80% 的 key 要点

# ── T-12h ── 叙事与公司功课（见模块 04）
#   · 四个 10 分钟故事各讲一遍（录音），检查 A 段是否占 >= 50%
#   · XPENG 技术路线：XNGP / 图灵芯片 / 端到端大模型 + VLA
#   · 想清「TSR 的输出交给谁、以什么形式交」——主动说出来是强信号
#   · 准备 5 个反问，其中 3 个必须是「不懂这行问不出来」的：
#       「你们的评测按什么切片做？有没有分像素尺寸的桶？」
#       「离线指标和路测体验背离的情况多吗？通常是哪一类原因？」
#       「一个 badcase 从发现到修复上线，典型周期多久？」

# ── 面试中的三条纪律 ──
#   ① 任何题都用：**判断依据 -> 方案 -> 验证**，讲完停下来交还控制权
#   ② 任何数字都带口径：mAP 报定义、延迟报 batch/卡型/含不含 NMS
#   ③ 不会就用三段式：承认边界 -> 给相邻知识 -> 给行动方案
#      **绝不用「应该是 / 大概」把不确定说成确定**

# ── 面试后（当天）──
#   · 三句话跟进邮件：感谢 / 重申最匹配的一点 / **补一个当时答得不好的完整答案**
#   · 自己复盘：卡住的题卡在三层追问的哪一层（口径 / 方差 / 机制），补完写进题库
'''
print(RECIPE)
for token in ['稳定排序', '每个 GT 只匹配一次', '205x', 'r_c = max(1, sqrt(t/f_c))',
              'prior bias', 'drill(5)', '判断依据 -> 方案 -> 验证', '分像素尺寸的桶']:
    assert token in RECIPE, token
print('✅ 清单覆盖：白板默写 / 关键数字 / 题库三轮 / 叙事与公司功课 / 面试中纪律 / 面试后跟进')

### 小结

- **白板题考的不是「会不会写」，是你写代码时暴露的工程习惯。**
  先口述思路 + 复杂度 + 边界情况清单，再动手；写完自己跑两个例子；
  最后补一句「这段代码在真实系统里的位置」。
- **五个必须默写的实现**：IoU（clip / 分母保护 / N×M 广播）、
  NMS（**稳定排序** / class-wise offset）、mAP（**每个 GT 只匹配一次** /
  recall 分母是 GT 数 / 末尾补 (1,0) / 单调包络）、匈牙利（贪心反例
  `[[1,2],[2,100]]`：101 vs 4）、Focal（稳定写法 + prior bias 初始化）。
- **同一条 PR 曲线，三种 AP 口径给出三个数**（本 notebook：5/9、6/11、56/101）；
  同一组预测，COCO mAP@[.5:.95] 比 mAP@0.5 低 24%。**报数必须报口径。**
- **Focal 的全部作用是「压易不压难」的比例差**：logit=−6 的易分负样本损失被压到
  $6\times10^{-6}$ 倍、梯度被压到 $1.8\times10^{-5}$ 倍，而 logit=0 的难样本只被压到 0.25 倍。
- **答题结构：判断依据 → 方案 → 验证，讲完停下来。**
  多数人的回答只有中间一段 —— 听不出为什么这么选，也听不出怎么证明它有用。
- **反问是最后一次展示判断力的机会。**判据和整门课一致：
  问一个「不了解这个领域就问不出来」的问题。

到这里，C53–C61 九门课的内容全部落到了两样东西上：
**四个可讲 10 分钟的深度故事**（模块 04）和**22 道能出声答完的题**（本模块）。
剩下的事只有一件 —— 把它们练到不用想。